# Microsoft Fabric - Restore Fabric Items from OneLake Files using Fabric CLI (Python)

This notebook uses the official Microsoft Fabric CLI (`fab`) from Python to export every exportable item from one or more workspaces.
- Idea for this based off the blog post: https://peerinsights.emono.dk/fabric-cli-beyond-shell-commands?s=03

In [ ]:
#########################
## Input Parameters
#########################

# This is the Workspace Name for the items you want to download - NOTE: It has got the .Workspace at the end as this is required for the Fabric CLI
target_workspaceName = "FILL ME IN.Workspace"

# This is the Workspace GUID
target_workspace_id = "FILL ME IN" 

# Lakehouse Name
target_lakehouse = "FILL ME IN"          

# Lakehouse GUID
target_lakehouse_id = "FILL ME IN"

# Backup Folder Location
restore_folder_location = "FILL ME IN"

In [ ]:
# Cell 1: Install Fabric CLI (only needed once per environment)
import sys
!{sys.executable} -m pip install --quiet ms-fabric-cli --upgrade
print("Fabric CLI installed/updated")

In [ ]:
######################################################################################### 
# Read secretes from Azure Key Vault
#########################################################################################
## This is the name of my Azure Key Vault 
key_vault = "FILL ME IN"
## I have stored my tenant id as one of the secrets to make it easier to use when needed 
tenant = notebookutils.credentials.getSecret(key_vault , "FILL ME IN") 
## This is my application Id for my service principal account 
client = notebookutils.credentials.getSecret(key_vault , "FILL ME IN") 
## This is my Client Secret for my service principal account 
client_secret = notebookutils.credentials.getSecret(key_vault , "FILL ME IN")


######################################################################################### 
# Authentication - Replace string variables with your relevant values 
#########################################################################################  

import json, requests, pandas as pd 
import datetime  

try: 
    from azure.identity import ClientSecretCredential 
except Exception:
     !pip install azure.identity 
     from azure.identity import ClientSecretCredential 

# Generates the access token for the Service Principal 
api = 'https://analysis.windows.net/powerbi/api/.default' 
auth = ClientSecretCredential(authority = 'https://login.microsoftonline.com/', 
               tenant_id = tenant, 
               client_id = client, 
               client_secret = client_secret) 
access_token = auth.get_token(api)
access_token = access_token.token 

## This is where I store my header with the Access Token, because this is required when authenticating 
## to the Power BI Admin APIs 
header = {'Authorization': f'Bearer {access_token}'}  

print('\nSuccessfully authenticated.')

In [ ]:
!fab config set encryption_fallback_enabled true

In [ ]:

from argparse import Namespace
from fabric_cli.commands.config import fab_config
from fabric_cli.commands.auth import fab_auth

# Login using service principal
args = Namespace(
    auth_command="login",
    username=client,
    password=client_secret,
    tenant=tenant,
    identity=None,
    federated_token=None,
    certificate=None
)
fab_auth.init(args)
fab_auth.status(None)  # Check current authentication status


In [ ]:
# Set App Workspace
import subprocess

workspace_name = target_workspaceName

result = subprocess.run(
    ["fab", "cd", workspace_name],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
# Restore Items - Single or Multiple
"""
Fabric Batch Restore Script
Restores single or multiple items from a backup location to a Fabric workspace.

Expected folder structure:
/lakehouse/default/Files/Restore/
└── Level1/
    └── MyNotebook.Notebook/    ← item name (level 2)
        └── BackupContents/     ← level 3, passed to fab import

Usage:
    # Restore all items (auto-discover)
    results = restore_all_items(WORKSPACE_NAME, BACKUP_ROOT)
    
    # Restore a single item by name
    results = restore_all_items(WORKSPACE_NAME, BACKUP_ROOT, single_item="MyNotebook.Notebook")
"""

import subprocess
from pathlib import Path


def restore_item(workspace_name: str, item_name: str, local_path: str) -> dict:
    """Restore a single item to a Fabric workspace."""
    result = subprocess.run(
        ["fab", "import", f"{workspace_name}/{item_name}", "-i", local_path, "-f"],
        capture_output=True,
        text=True
    )
    return {
        "item": item_name,
        "success": result.returncode == 0,
        "stdout": result.stdout,
        "stderr": result.stderr
    }


def discover_items(backup_root: str, single_item: str = None) -> list[dict]:
    """
    Discover items in the three-level backup structure.
    
    Args:
        backup_root: Root directory containing backups
        single_item: If provided, only find and return this specific item.
                     Can be just the item name (e.g., "MyNotebook.Notebook")
                     or a partial path match.
    
    Returns:
        List of dicts with 'item_name' and 'restore_path' keys.
    """
    items = []
    backup_path = Path(backup_root)
    
    # Level 1: iterate through first-level folders
    for level1 in backup_path.iterdir():
        if not level1.is_dir():
            continue
        
        # Level 2: item name folders
        for level2 in level1.iterdir():
            if not level2.is_dir():
                continue
            
            item_name = level2.name  # e.g., "MyNotebook.Notebook"
            
            # If single_item specified, skip non-matching items
            if single_item is not None:
                # Match by exact name or partial match (case-insensitive)
                if not (item_name == single_item or 
                        single_item.lower() in item_name.lower()):
                    continue
            
            # Level 3: find the actual backup content folder
            for level3 in level2.iterdir():
                if level3.is_dir():
                    items.append({
                        "item_name": item_name,
                        "restore_path": str(level3)
                    })
                    break  # Assume one level3 folder per item
            
            # If we found the single item we're looking for, stop searching
            if single_item is not None and items:
                return items
    
    return items


def find_item_by_name(backup_root: str, item_name: str) -> dict | None:
    """
    Find a specific item by name in the backup structure.
    
    Args:
        backup_root: Root directory containing backups
        item_name: The item name to find (e.g., "MyNotebook.Notebook")
    
    Returns:
        Dict with 'item_name' and 'restore_path', or None if not found.
    """
    items = discover_items(backup_root, single_item=item_name)
    return items[0] if items else None


def list_available_items(backup_root: str) -> list[str]:
    """
    List all available item names in the backup structure.
    
    Args:
        backup_root: Root directory containing backups
    
    Returns:
        List of item names.
    """
    items = discover_items(backup_root)
    return [item["item_name"] for item in items]


def restore_single_item(workspace_name: str, backup_root: str, item_name: str) -> dict:
    """
    Restore a single specific item to a Fabric workspace.
    
    Args:
        workspace_name: Target workspace (e.g., "Adam Fabric Playpen.Workspace")
        backup_root: Root directory containing backups
        item_name: The specific item to restore (e.g., "MyNotebook.Notebook")
    
    Returns:
        Result dictionary for the item.
    """
    item = find_item_by_name(backup_root, item_name)
    
    if item is None:
        available = list_available_items(backup_root)
        return {
            "item": item_name,
            "success": False,
            "stdout": "",
            "stderr": f"Item '{item_name}' not found. Available items: {available}"
        }
    
    print(f"Restoring: {item['item_name']}...", end=" ")
    result = restore_item(workspace_name, item["item_name"], item["restore_path"])
    
    if result["success"]:
        print("✓")
    else:
        print(f"❌ Error: {result['stderr']}")
    
    return result


def restore_all_items(
    workspace_name: str, 
    backup_root: str, 
    items: list[dict] = None,
    single_item: str = None
) -> list[dict]:
    """
    Restore items to a Fabric workspace.
    
    Args:
        workspace_name: Target workspace (e.g., "Adam Fabric Playpen.Workspace")
        backup_root: Root directory containing backups
        items: Optional list of item dicts with 'item_name' and 'restore_path'.
               If None, auto-discovers from backup_root.
        single_item: If provided, only restore this specific item by name.
                     Overrides the 'items' parameter.
    
    Returns:
        List of result dictionaries for each item.
    """
    results = []
    
    # Handle single item mode
    if single_item is not None:
        result = restore_single_item(workspace_name, backup_root, single_item)
        return [result]
    
    # Handle batch mode
    if items is None:
        items = discover_items(backup_root)
    
    if not items:
        print("No items found to restore.")
        return results
    
    print(f"Restoring {len(items)} items to {workspace_name}")
    print("-" * 50)
    
    for item in items:
        item_name = item["item_name"]
        restore_path = item["restore_path"]
        
        print(f"Restoring: {item_name}...", end=" ")
        result = restore_item(workspace_name, item_name, restore_path)
        results.append(result)
        
        if result["success"]:
            print("✓")
        else:
            print(f"❌ Error: {result['stderr']}")
    
    return results


def print_summary(results: list[dict]):
    """Print a summary of restore operations."""
    successful = sum(1 for r in results if r["success"])
    failed = len(results) - successful
    
    print("\n" + "=" * 50)
    print(f"RESTORE SUMMARY: {successful} succeeded, {failed} failed")
    print("=" * 50)
    
    if failed > 0:
        print("\nFailed items:")
        for r in results:
            if not r["success"]:
                print(f"  - {r['item']}: {r['stderr']}")


if __name__ == "__main__":
    # Configuration
    WORKSPACE_NAME = target_workspaceName
    BACKUP_ROOT = restore_folder_location
    
    # Option 1: Restore ALL items (auto-discover)
    results = restore_all_items(WORKSPACE_NAME, BACKUP_ROOT)
    
    # Option 2: Restore a SINGLE item by name
    # SINGLE_ITEM_NAME = "A.RunPerfScenario.Notebook"  # Change this to your item name
    # results = restore_all_items(WORKSPACE_NAME, BACKUP_ROOT, single_item=SINGLE_ITEM_NAME)
    
    # Option 3: List available items first, then restore one
    # available = list_available_items(BACKUP_ROOT)
    # print("Available items:", available)
    # results = restore_single_item(WORKSPACE_NAME, BACKUP_ROOT, available[0])
    
    print_summary(results)